# Notebook 07 — Displacement Null vs Electrostatic Blind Spot

## Physical question

In contact-resonance AFM / PFM two distinct positions along the cantilever
produce zero **displacement** *z*(*x*_obs):

| Null | Frequency | Origin |
|------|-----------|--------|
| **Displacement null** | at *f*_CR (contact resonance) | Spatial node of the resonance mode shape |
| **Electrostatic blind spot** | quasi-static (*f* ≪ *f*_CR) | Body-electrostatic forcing produces zero displacement |

**Are these at the same position?**

The resonance null is set by the eigenfunction of the beam+contact system,
including frequency-dependent inertia corrections (terms *T*, *U* in the Bradler BCs).
The quasi-static ES blind spot is governed by the convolution of the distributed
electrostatic load *q*(*x*) with the quasi-static Green’s function — a different object.

## Experiment design

1. **Part 1 — 2D spectrogram**: sweep frequency around *f*_CR and observation
   position over the last 50 µm. Log-scale (dB) displacement heatmap reveals
   the resonance amplitude minimum near the tip and anti-resonance structure.
2. **Part 2 — DC-bias sweep at 30 kHz**: for each position vary *ΔV* at fixed
   *V*_AC and record Re(*z*). The slope of each line = local ES sensitivity.
   Find the position of minimum sensitivity (electrostatic blind spot).
3. **Comparison**: overlay both profiles and extract the offset Δ*x*.

**Setup**: Bradler 2016 cantilever, stiff contact (near-pinned),
interferometric displacement detection *z*(*x*_obs).

**Solver**: all results use the **two-segment overhang model** (commit `3aceb6e`) —
the tip contact acts at x_c = L − tip_offset through interior jump conditions and the
overhang beyond it is a genuinely free segment (z″(L) = z‴(L) = 0) that also carries
the distributed electrostatic load. The overhang is the part of the lever closest to
the sample, so it takes the largest electrostatic load per unit length; this
qualitatively changes the low-stiffness electrostatic blind spot compared to the
legacy terminal-BC treatment used in earlier versions of this notebook.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../src'))

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from eb_cr_afm import (
    CantileverParams, ContactParams, ExcitationParams, SolverOptions,
    frequency_sweep, solve_single_frequency,
)

RESULTS_DIR = os.path.join(os.getcwd(), '../results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print('eb_cr_afm loaded OK')
print(f'Results -> {os.path.abspath(RESULTS_DIR)}')

## Parameters  ← configure everything here

In [ ]:
# -----------------------------------------------------------------------
# DRIVE FLAGS  (toggle True / False)
# -----------------------------------------------------------------------
USE_PIEZO    = True    # substrate displacement drive d1 (acoustic/mechanical)
USE_BODY_ES  = True    # distributed body electrostatics q(x)
USE_LOCAL_ES = False   # local tip electrostatic force f0

# -----------------------------------------------------------------------
# CANTILEVER -- Bradler et al. (2016) Table I
# -----------------------------------------------------------------------
cant = CantileverParams(
    L=240e-6,   # m   total length
    w=40e-6,    # m   width
    H=6e-6,     # m   tip height
    k=3.0,      # N/m static spring constant
    m=5e-11,    # kg  total mass
    mT=5e-12,   # kg  tip mass
    phi=0.2,    # rad tilt angle (~11.5 deg)
    tip_offset=10e-6,
)

# -----------------------------------------------------------------------
# CONTACT -- stiff, near-pinned vertical BC
# -----------------------------------------------------------------------
K1 = 3000.0    # N/m  vertical contact stiffness  <- tune here
K2 = 0.85*K1  # N/m  lateral stiffness (Mindlin: k2 ~ 0.85*k1)
C1 = 20e-6     # N*s/m
C2 =  0.0      # N*s/m

contact = ContactParams(k1=K1, k2=K2, c1=C1, c2=C2, psi=0.0)

# -----------------------------------------------------------------------
# EXCITATION AMPLITUDES
# -----------------------------------------------------------------------
D1      = 1e-12   # m   substrate displacement amplitude (piezo)
V_AC    = 1.0     # V   AC voltage amplitude (body ES)
DELTA_V = 1.0     # V   DC contact potential (body ES reference)

# Combined excitation -- used in Part 1 spectrogram (respects flags above)
exc = ExcitationParams(
    d1  = D1 if USE_PIEZO else 0.0,
    d2  = 0.0,
    f0  = 0.0,
    use_f0            = USE_LOCAL_ES,
    bias_coupled_f0   = False,
    delta_V           = DELTA_V,
    V_AC              = V_AC,
    eps_r             = 1.0,
    use_distributed_load = USE_BODY_ES,
)

# Body-ES-only excitation -- Part 2 isolates ES sensitivity cleanly
exc_es = ExcitationParams(
    d1=0.0, d2=0.0, f0=0.0,
    use_f0=False, bias_coupled_f0=False,
    delta_V=DELTA_V, V_AC=V_AC, eps_r=1.0,
    use_distributed_load=True,
)

# -----------------------------------------------------------------------
# SOLVER
# -----------------------------------------------------------------------
N_POINTS = 600      # spatial grid density
G_AIR    = 20.0     # 1/s  air damping coefficient
opts = SolverOptions(n_points=N_POINTS, g=G_AIR, method='analytical')

# -----------------------------------------------------------------------
# SCAN GEOMETRY -- last 50 um of cantilever (toward tip)
# -----------------------------------------------------------------------
SCAN_WINDOW = 50e-6    # m
N_OBS       = 80
obs_pos  = np.linspace(cant.L - SCAN_WINDOW, cant.L, N_OBS)  # m from clamp
dist_tip = (cant.L - obs_pos) * 1e6                           # um from tip

# -----------------------------------------------------------------------
# QUASI-STATIC PROBE (Part 2)
# -----------------------------------------------------------------------
F_QUASI  = 30e3                         # Hz
DV_SWEEP = np.linspace(-5.0, 5.0, 21)  # V  DC bias sweep

In [ ]:
print('=' * 62)
print('  SIMULATION CONFIGURATION')
print('=' * 62)
print(f'  Cantilever : L={cant.L*1e6:.0f} um, k={cant.k} N/m, '
      f'phi={np.degrees(cant.phi):.1f} deg')
print(f'  Contact    : k1={K1:.0f} N/m  (kappa={K1/cant.k:.0f})')
print(f'  Piezo      : {"ON" if USE_PIEZO else "OFF"}  '
      f'(d1={(D1 if USE_PIEZO else 0)*1e12:.1f} pm)')
print(f'  Body ES    : {"ON" if USE_BODY_ES else "OFF"}  '
      f'(V_AC={V_AC} V, dV={DELTA_V} V)')
print(f'  Local ES   : {"ON" if USE_LOCAL_ES else "OFF"}')
print(f'  Scan       : last {SCAN_WINDOW*1e6:.0f} um, {N_OBS} positions')
print(f'  ES probe   : {F_QUASI/1e3:.0f} kHz | dV [{DV_SWEEP[0]:.0f},{DV_SWEEP[-1]:.0f}] V')
print('=' * 62)

## 0 — Contact resonance identification

Quick frequency sweep at a reference position 100 µm from the tip
to locate *f*_CR before building the 2D spectrogram.

In [ ]:
OBS_X_REF = cant.L - 100e-6    # reference obs_x: 100 um from tip

opts_q = SolverOptions(n_points=300, g=G_AIR, method='analytical',
                       f_start=200e3, f_stop=480e3, f_step=1e3)

print('Quick identification sweep ...')
sw_q = frequency_sweep(cant, contact, exc, opts_q,
                       obs_x=OBS_X_REF, verbose=False)

i_cr = int(np.argmax(sw_q['amplitudes_disp']))
F_CR_COARSE = sw_q['freqs'][i_cr]

# Refine the peak with a bounded 1-D optimiser: the resonance linewidth is only a
# few Hz at g = 20 s^-1, so any fixed frequency grid is too coarse (the previous
# version stopped at the 1 kHz grid point).
from scipy.optimize import minimize_scalar
opts_pt = SolverOptions(n_points=300, g=G_AIR, method='analytical')

def _neg_amp(f):
    return -solve_single_frequency(f, cant, contact, exc, opts_pt,
                                   obs_x=OBS_X_REF)['A_disp']

_ref = minimize_scalar(_neg_amp, bounds=(F_CR_COARSE - 1.2e3, F_CR_COARSE + 1.2e3),
                       method='bounded', options=dict(xatol=0.05))
F_CR = float(_ref.x)
print(f'  coarse peak {F_CR_COARSE/1e3:.1f} kHz  ->  refined f_CR = {F_CR/1e3:.3f} kHz'
      f'  | peak |z| = {-_ref.fun:.3e} m')

fig_cr = go.Figure()
fig_cr.add_trace(go.Scatter(
    x=sw_q['freqs'] * 1e-3, y=sw_q['amplitudes_disp'],
    mode='lines', line=dict(color='steelblue', width=2),
    name='|z(L-100 um)|',
))
fig_cr.add_vline(
    x=F_CR * 1e-3, line_dash='dash', line_color='red',
    annotation_text=f'f_CR = {F_CR/1e3:.1f} kHz',
)
fig_cr.update_layout(
    title='Identification sweep -- displacement at L - 100 um',
    xaxis_title='Frequency (kHz)',
    yaxis_title='|z| (m)',
    yaxis_type='log',
    template='plotly_white', height=340,
)
fig_cr.show()

---
## Part 1 — 2D Displacement Spectrogram

For each observation position *x*_obs in the last 50 µm and each frequency
near *f*_CR, the EB BVP is solved **once per frequency** and |*z*(*x*_obs, *f*)|
is extracted from the full spatial profile (one solve = all 80 obs positions).

**Log-scale (dB) heatmap** reveals:
- **Bright band**: contact resonance ridge
- **Dark notch** near the tip (0 µm): mode amplitude approaching zero
  → displacement null of mode 1 under the near-pinned BC
- **Anti-resonance structure**: dark regions flanking the resonance at
  positions where piezo and body-ES contributions interfere destructively
- **Phase panel**: 180° jump marks the node crossing

y-axis convention: 0 µm = tip (*x* = *L*), 50 µm = base of scan window.

In [ ]:
F_SPAN    = 80e3     # Hz  half-bandwidth around f_CR
F_STEP_2D = 600.0    # Hz  frequency step
freqs_2d  = np.arange(F_CR - F_SPAN,
                       F_CR + F_SPAN + F_STEP_2D * 0.5,
                       F_STEP_2D)
N_FREQ = len(freqs_2d)

A_2d  = np.zeros((N_OBS, N_FREQ))
ph_2d = np.zeros((N_OBS, N_FREQ))

print(f'2D sweep: {N_OBS} positions x {N_FREQ} frequencies '
      f'({freqs_2d[0]/1e3:.0f}-{freqs_2d[-1]/1e3:.0f} kHz) ...')

for i_f, f in enumerate(freqs_2d):
    if i_f % 50 == 0:
        print(f'  {f/1e3:.0f} kHz  ({i_f+1}/{N_FREQ})')
    res = solve_single_frequency(f, cant, contact, exc, opts)
    xg, zg = res['x'], res['z']   # already includes overhang if tip_offset > 0
    for i_o, xo in enumerate(obs_pos):
        zr = float(np.interp(xo, xg, np.real(zg)))
        zi = float(np.interp(xo, xg, np.imag(zg)))
        zv = zr + 1j * zi
        A_2d[i_o, i_f]  = np.abs(zv)
        ph_2d[i_o, i_f] = np.degrees(np.angle(zv))

print('Done.')

In [ ]:
# A_2d shape: (N_OBS, N_FREQ)  ->  transpose to (N_FREQ, N_OBS) for heatmap
x_um      = obs_pos * 1e6                                   # position from clamp
freqs_kHz = freqs_2d * 1e-3
z_amp_log = np.log10(np.clip(A_2d.T, 1e-17, None))         # log10(m)
z_phase   = ph_2d.T                                         # degrees

fig2d = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '|z(x,f)| amplitude (log₁₀ m)',
        'z(x,f) phase (°)',
    ],
    horizontal_spacing=0.15,
)

fig2d.add_trace(go.Heatmap(
    x=x_um, y=freqs_kHz, z=z_amp_log,
    colorscale='RdBu_r',
    colorbar=dict(title='log₁₀(m)', x=0.44, len=0.85),
    hovertemplate=(
        'x = %{x:.1f} µm<br>f = %{y:.1f} kHz<br>log A = %{z:.2f}<extra></extra>'
    ),
), row=1, col=1)

fig2d.add_trace(go.Heatmap(
    x=x_um, y=freqs_kHz, z=z_phase,
    colorscale='Plasma', zmin=-180, zmax=180,
    colorbar=dict(title='Phase (°)', x=1.01, len=0.85),
    hovertemplate=(
        'x = %{x:.1f} µm<br>f = %{y:.1f} kHz<br>φ = %{z:.1f}°<extra></extra>'
    ),
), row=1, col=2)

# Mark f_CR as horizontal line (frequency on y-axis)
for col in (1, 2):
    fig2d.add_hline(
        y=F_CR * 1e-3,
        line_dash='dash', line_color='lime', line_width=1.2,
        annotation_text=f'f_CR={F_CR/1e3:.0f} kHz',
        annotation_font_color='lime',
        row=1, col=col,
    )

for col in (1, 2):
    fig2d.update_xaxes(title_text='Position x (µm)', row=1, col=col)
    fig2d.update_yaxes(title_text='Frequency (kHz)', row=1, col=col)

drive_label = '+'.join(filter(None, [
    'Piezo'   if USE_PIEZO    else '',
    'BodyES'  if USE_BODY_ES  else '',
    'LocalES' if USE_LOCAL_ES else '',
])) or 'none'

fig2d.update_layout(
    height=480, width=1120,
    template='plotly_white',
    title=(
        f'2D displacement spectrogram | last {SCAN_WINDOW*1e6:.0f} µm | '
        f'drive: {drive_label} | k1={K1:.0f} N/m'
    ),
)
fig2d.show()

fname = os.path.join(RESULTS_DIR, 'nb07_2d_spectrogram')
fig2d.write_html(fname  + '.html')
print(f'Saved: {os.path.basename(fname)}.html')

### Resonance null — amplitude profile at *f*_CR

Slice the 2D array at *f* = *f*_CR to get the 1D amplitude profile.
A parabolic fit around the discrete minimum gives the sub-grid null position.
With the two-segment solver the on-resonance null is a true interior node on the
overhang for stiff contact.

In [ ]:
i_fcr = int(np.argmin(np.abs(freqs_2d - F_CR)))
A_res = A_2d[:, i_fcr]   # amplitude profile along x_obs at f_CR

# 1D profile plot
fig_res1d = go.Figure()
fig_res1d.add_trace(go.Scatter(
    x=dist_tip, y=A_res * 1e12,
    mode="lines+markers", marker=dict(size=4),
    line=dict(color="#1f4e79", width=2.5),
    name=f"|z(x_obs)| at f_CR={F_CR/1e3:.0f} kHz",
))
fig_res1d.update_layout(
    title=f"Displacement amplitude vs obs. position at f_CR = {F_CR/1e3:.1f} kHz",
    xaxis=dict(title="Distance from tip (um)", autorange="reversed"),
    yaxis_title="|z(x_obs)| (pm)",
    template="plotly_white", height=360, width=700,
)
fig_res1d.show()

# Parabolic fit around the minimum (same approach as ES blind spot)
i_res_min = int(np.argmin(A_res))
x_res_scan_um = dist_tip[i_res_min]

WIN = 5
i_lo = max(0, i_res_min - WIN)
i_hi = min(N_OBS - 1, i_res_min + WIN)
x_fit_r = dist_tip[i_lo:i_hi + 1]
A_fit_r  = A_res[i_lo:i_hi + 1]

if len(x_fit_r) >= 3:
    c_para_r = np.polyfit(x_fit_r, A_fit_r, 2)
    a_r, b_r, _ = c_para_r
    if a_r > 0:
        x_null_res_um = -b_r / (2 * a_r)
    else:
        x_null_res_um = x_res_scan_um
else:
    x_null_res_um = x_res_scan_um

# Clamp to scan window
x_null_res_um = float(np.clip(x_null_res_um, dist_tip.min(), dist_tip.max()))

print(f"At f_CR = {F_CR/1e3:.1f} kHz:")
print(f"  |z(tip)|           = {A_res[-1]:.3e} m")
print(f"  |z(L - 50 um)|    = {A_res[0]:.3e} m")
print(f"  Discrete minimum at:   {x_res_scan_um:.2f} um from tip")
print(f"  Parabolic fit minimum: {x_null_res_um:.4f} um from tip")

---
## Part 2 — Electrostatic Blind Spot at 30 kHz

At *f* = 30 kHz (quasi-static, ≪ *f*_CR) with **body electrostatics only**
(piezo off — isolating the ES response), the displacement scales linearly:

$$z(x_{\text{obs}},\,30\,\text{kHz},\,\Delta V)
  = G(x_{\text{obs}}) \cdot V_{\text{AC}} \cdot \Delta V$$

A DC-bias sweep (*ΔV* swept, *V*_AC fixed) produces straight lines through
the origin. The **slope** = Re[*G*(*x*_obs)] is the local ES sensitivity.
The **electrostatic blind spot** is where |*G*| is minimised.

**Two-segment caveat**: the overhang is the part of the lever closest to the sample
(end gap ≈ 3.9 µm vs ≈ 5.9 µm above the contact for this geometry, i.e. ≈2.3× the
load per unit length), so once the electrostatic load on the overhang is included
the interior blind spot **does not exist at soft-to-moderate contact** — the ES
response then decreases monotonically toward the tip without a zero, and the
extracted "blind spot" is just the profile minimum at the scan edge. For the
default k₁ = 3 kN/m (Mindlin k₂) this is exactly what happens; an interior ESBS
first appears just above, near k₁ ≈ 4 kN/m (initially within 0.5 µm of the tip,
then migrating inward — Part 4).

In [ ]:
opts_es = SolverOptions(n_points=N_POINTS * 2, g=G_AIR, method='analytical')

print(f'Solving at f = {F_QUASI/1e3:.0f} kHz  '
      f'(body ES only, V_AC={V_AC} V, dV={DELTA_V} V) ...')
res_30 = solve_single_frequency(F_QUASI, cant, contact, exc_es, opts_es)

xg30 = res_30['x']   # already includes overhang if tip_offset > 0
zg30 = res_30['z']

# Complex displacement at each obs position
z_es_obs = np.array([
    float(np.interp(xo, xg30, np.real(zg30)))
    + 1j * float(np.interp(xo, xg30, np.imag(zg30)))
    for xo in obs_pos
])
A_es_obs = np.abs(z_es_obs)

print(f'  |z(tip)|          = {A_es_obs[-1]:.3e} m')
print(f'  |z(L - 50 um)|   = {A_es_obs[0]:.3e} m')
print(f'  Ratio tip/edge    = {A_es_obs[-1]/A_es_obs[0]:.4f}')

In [ ]:
# Overlay: resonance profile scaled to same amplitude at L - 50 um
scale_res = A_es_obs[0] / (A_res[0] + 1e-300)

fig_sp = go.Figure()
fig_sp.add_trace(go.Scatter(
    x=dist_tip, y=A_res * scale_res * 1e12,
    mode='lines', line=dict(color='#1f4e79', width=2.5, dash='dot'),
    name=f'Resonance ({F_CR/1e3:.0f} kHz) -- amplitude-scaled',
))
fig_sp.add_trace(go.Scatter(
    x=dist_tip, y=A_es_obs * 1e12,
    mode='lines', line=dict(color='#c00000', width=2.5),
    name=f'Quasi-static ES ({F_QUASI/1e3:.0f} kHz)',
))
fig_sp.update_layout(
    title='Spatial displacement sensitivity -- resonance vs quasi-static ES',
    xaxis=dict(title='Distance from tip (um)', autorange='reversed'),
    yaxis_title='|z(x_obs)| (pm)',
    template='plotly_white', height=380, width=740,
    legend=dict(x=0.35, y=0.98, font_size=11),
)
fig_sp.show()

fname = os.path.join(RESULTS_DIR, 'nb07_es_spatial')
fig_sp.write_html(fname  + '.html')
print(f'Saved: {os.path.basename(fname)}.html')

In [ ]:
# Six representative positions spanning the last 50 um
sel_idx  = np.round(np.linspace(0, N_OBS - 1, 6)).astype(int)
sel_dtip = dist_tip[sel_idx]   # um from tip
sel_xobs = obs_pos[sel_idx]    # m

COLORS = ['#1a3658', '#2e75b6', '#4bacc6', '#70ad47', '#ed7d31', '#c00000']

fig_bias = go.Figure()

for xo, dtip, col in zip(sel_xobs, sel_dtip, COLORS):
    # z(dV) = Re(z_base) * dV / DELTA_V   (V_AC = 1 V locked in exc_es)
    z_base_re  = float(np.interp(xo, xg30, np.real(zg30)))
    z_re_sweep = z_base_re * DV_SWEEP / DELTA_V  # m

    fig_bias.add_trace(go.Scatter(
        x=DV_SWEEP,
        y=z_re_sweep * 1e12,   # pm
        mode='lines+markers',
        marker=dict(size=5),
        line=dict(color=col, width=2),
        name=f'dist from tip = {dtip:.0f} um',
    ))

fig_bias.add_hline(y=0, line_dash='dash', line_color='gray', line_width=0.8)
fig_bias.add_vline(x=0, line_dash='dash', line_color='gray', line_width=0.8)

fig_bias.update_layout(
    title=(
        f'DC-bias sweep at {F_QUASI/1e3:.0f} kHz  --  '
        f'Re[z(x_obs)] vs dV  (V_AC={V_AC} V, body ES only)'
    ),
    xaxis_title='dV  (V)',
    yaxis_title='Re[z(x_obs, 30 kHz)]  (pm)',
    template='plotly_white', height=460, width=740,
    legend=dict(x=0.02, y=0.98, font_size=11),
)
fig_bias.show()

fname = os.path.join(RESULTS_DIR, 'nb07_bias_sweep')
fig_bias.write_html(fname  + '.html')
print(f'Saved: {os.path.basename(fname)}.html')

In [ ]:
i_es_min     = int(np.argmin(A_es_obs))
x_es_scan_um = dist_tip[i_es_min]
es_min_at_edge = (i_es_min == 0) or (i_es_min == N_OBS - 1)

# ── Parabolic fit around the minimum for sub-grid precision ─────────────────
# Use a window of ±5 points around the discrete minimum, clamped to array bounds
WIN = 5
i_lo = max(0, i_es_min - WIN)
i_hi = min(N_OBS - 1, i_es_min + WIN)
x_fit = dist_tip[i_lo:i_hi + 1]
A_fit = A_es_obs[i_lo:i_hi + 1]

if len(x_fit) >= 3:
    c_para = np.polyfit(x_fit, A_fit, 2)          # a*x^2 + b*x + c
    a_p, b_p, _ = c_para
    if a_p > 0:                                    # proper minimum (concave up)
        x_null_ES_um = -b_p / (2 * a_p)
    else:                                          # fallback: use discrete minimum
        x_null_ES_um = x_es_scan_um
else:
    x_null_ES_um = x_es_scan_um

# Clamp to scan window for reporting (parabola can wander if fit is poor)
x_null_ES_um_clamped = float(np.clip(x_null_ES_um,
                                      dist_tip.min(), dist_tip.max()))

if es_min_at_edge:
    print('NOTE: the ES minimum sits at the scan edge -> there is NO interior ES')
    print('      blind spot in the last 50 um at this k1. The two-segment solver')
    print('      applies the electrostatic load to the overhang (the part of the')
    print('      lever closest to the sample), which removes the interior blind')
    print('      spot at soft-to-moderate contact; x_ES below is the profile')
    print('      minimum at the tip, not a true null.')
    print()

print(f'At {F_QUASI/1e3:.0f} kHz (body ES only):')
print(f'  Discrete minimum at:  {x_es_scan_um:.2f} um from tip')
print(f'  Parabolic fit minimum: {x_null_ES_um:.4f} um from tip')
if abs(x_null_ES_um - x_null_ES_um_clamped) > 0.01:
    print(f'  (clamped to scan window: {x_null_ES_um_clamped:.4f} um)')
    x_null_ES_um = x_null_ES_um_clamped

Dx = x_null_ES_um - x_null_res_um

print()
print('-' * 58)
print(f'  Resonance null  (f_CR={F_CR/1e3:.0f} kHz):  {x_null_res_um:+.4f} um from tip')
print(f'  ES blind spot   ({F_QUASI/1e3:.0f} kHz):       {x_null_ES_um:+.4f} um from tip')
print(f'  Offset Dx = x_ES - x_res = {Dx:+.4f} um')
print('-' * 58)
if abs(Dx) < 0.05:
    print('  -> Positions coincident within fit precision.')
elif Dx < 0:
    print(f'  -> ES blind spot is {abs(Dx):.4f} um CLOSER to tip than res. null.')
else:
    print(f'  -> ES blind spot is {abs(Dx):.4f} um FURTHER from tip than res. null.')

---
## Part 3 — Comparison

Both normalised profiles on one figure with the extracted minimum positions marked.
A second panel shows d|*A*|/d(Δtip) — the spatial decay rate.
At the default k₁ = 3 kN/m the resonance profile has a genuine interior node
(≈7.7 µm from the tip, on the overhang) while the ES profile has **no interior
zero** — its minimum is at the physical tip — so Δ*x* here measures the distance
from the resonance node to the free end, not a null-to-null separation.

In [ ]:
A_res_n = A_res    / (A_res[0]    + 1e-300)
A_es_n  = A_es_obs / (A_es_obs[0] + 1e-300)

fig_cmp = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Normalised amplitude profiles in last 50 um',
        'd|A|/d(dist_tip)  -- spatial decay rate',
    ],
    horizontal_spacing=0.14,
)

fig_cmp.add_trace(go.Scatter(
    x=dist_tip, y=A_res_n,
    mode='lines', line=dict(color='#1f4e79', width=2.5),
    name=f'Resonance ({F_CR/1e3:.0f} kHz)',
), row=1, col=1)

fig_cmp.add_trace(go.Scatter(
    x=dist_tip, y=A_es_n,
    mode='lines', line=dict(color='#c00000', width=2.5, dash='dot'),
    name=f'ES probe ({F_QUASI/1e3:.0f} kHz)',
), row=1, col=1)

# Mark extrapolated null positions
for xn, lbl, col in [
    (x_null_res_um, f'Res. null ({x_null_res_um:+.2f} um)', '#1f4e79'),
    (x_null_ES_um,  f'ES blind  ({x_null_ES_um:+.2f} um)',  '#c00000'),
]:
    fig_cmp.add_shape(
        type='line',
        x0=xn, x1=xn, y0=-0.05, y1=1.15,
        line=dict(color=col, dash='dash', width=1.5),
        row=1, col=1,
    )
    fig_cmp.add_annotation(
        x=xn + 0.5, y=1.12,
        text=lbl, showarrow=False,
        font=dict(color=col, size=10),
        row=1, col=1,
    )

fig_cmp.update_xaxes(
    title_text='Distance from tip (um)  [0 = tip]',
    autorange='reversed',
)
fig_cmp.update_yaxes(
    title_text='A / A(L - 50 um)', row=1, col=1,
    range=[-0.05, 1.25],
)

g_res = np.gradient(A_res_n, dist_tip)
g_es  = np.gradient(A_es_n,  dist_tip)

fig_cmp.add_trace(go.Scatter(
    x=dist_tip, y=g_res,
    mode='lines', line=dict(color='#1f4e79', width=2),
    showlegend=False,
), row=1, col=2)

fig_cmp.add_trace(go.Scatter(
    x=dist_tip, y=g_es,
    mode='lines', line=dict(color='#c00000', width=2, dash='dot'),
    showlegend=False,
), row=1, col=2)

fig_cmp.add_hline(y=0, line_dash='dash', line_color='gray',
                  line_width=0.8, row=1, col=2)
fig_cmp.update_yaxes(title_text='dA_norm/d(dist_tip)  (1/um)', row=1, col=2)

fig_cmp.update_layout(
    height=460, width=1140,
    template='plotly_white',
    title=f'Resonance null vs ES blind spot | k1={K1:.0f} N/m | Dx={Dx:+.4f} um',
    legend=dict(x=0.53, y=0.98, font_size=12),
)
fig_cmp.show()

fname = os.path.join(RESULTS_DIR, 'nb07_comparison')
fig_cmp.write_html(fname  + '.html')
print(f'Saved: {os.path.basename(fname)}.html')

In [ ]:
print()
print('=' * 62)
print('  RESULTS SUMMARY')
print('=' * 62)
tip_offset_um = cant.tip_offset * 1e6
contact_x_um  = cant.contact_x * 1e6
print(f'  Cantilever  : L={cant.L*1e6:.0f} um, tip_offset={tip_offset_um:.1f} um')
print(f'  Contact pt  : {contact_x_um:.1f} um from clamp  =  {tip_offset_um:.1f} um from physical tip')
print(f'  k1          = {K1:.0f} N/m  (kappa={K1/cant.k:.0f})')
print(f'  f_CR        = {F_CR/1e3:.2f} kHz')
print()
print(f'  |z(phys. tip, x=L)| at f_CR       = {A_res[-1]:.3e} m')
print(f'  |z(phys. tip, x=L)| at {F_QUASI/1e3:.0f} kHz (ES) = {A_es_obs[-1]:.3e} m')
print()
x_null_res_from_contact = x_null_res_um - tip_offset_um
x_null_ES_from_contact  = x_null_ES_um  - tip_offset_um
print(f'  Resonance null:')
print(f'    {x_null_res_um:+.4f} um from physical tip (x=L)')
print(f'    {x_null_res_from_contact:+.4f} um from contact point  (- = past contact, on overhang; + = before contact)')
print()
print(f'  ES blind spot:')
print(f'    {x_null_ES_um:+.4f} um from physical tip (x=L)')
print(f'    {x_null_ES_from_contact:+.4f} um from contact point  (- = past contact, on overhang; + = before contact)')
print()
print(f'  Offset Dx = x_ES - x_res = {Dx:+.4f} um')
print()
if abs(Dx) < 0.05:
    interp = 'Positions coincident within fit precision'
elif Dx < 0:
    interp = 'ES blind spot sits CLOSER to physical tip than resonance null'
else:
    interp = 'ES blind spot sits FURTHER from physical tip than resonance null'
print(f'  {interp}')
print('=' * 62)


---
## Part 4 — Null positions vs contact stiffness (load)

Sweep k₁ from 100 N/m to 3 MN/m (increasing contact load, since Hertz contact gives
k₁ ∝ F_N^{1/3}). At each k₁:
1. Locate f_CR by a coarse 1 kHz frequency sweep, then refine the peak with a bounded
   `minimize_scalar` (the resonance linewidth is only ~Hz at g = 20 s⁻¹, so a fixed
   grid is far too coarse at high k₁)
2. Extract the resonance null position from the displacement profile at f_CR
3. Extract the ES blind spot from the 30 kHz profile

Left panel: both null positions vs k₁, with the contact point marked.
Right panel: Δx = x_ES − x_res vs k₁ (positive = ES further from tip).
Points where the minimum is at the scan edge or shallower than 3 % of the window
maximum (no true null — expected for the ES profile at soft contact, where the
overhang load removes the interior blind spot) are shown faded.

In [ ]:
# -----------------------------------------------------------------------
# Contact stiffness (load) sweep
# k1 ~ (6 E* R F_N)^(1/3) so sweeping k1 approximates sweeping load
#
# K2_FACTOR mirrors the K2/K1 ratio set in the params cell so that
# changing K2 there is immediately reflected in this sweep.
#
# NOTE: det_bc_matrix is NOT used here because |U| ~ k1/EI ~ 10^15 at
# high k1, making the 2x2 BC matrix numerically singular (12 orders of
# magnitude between T and U rows). Instead we locate f_CR from the peak
# of |z(obs_x, f)|, then polish it with a bounded minimize_scalar: the
# resonance linewidth is only ~Hz at g = 20 s^-1, so a fixed frequency
# grid alone is far too coarse at high k1.
# The search window is seeded adaptively from the previous k1's f_CR so
# we always track the same mode branch.
# -----------------------------------------------------------------------
K2_FACTOR = K2 / K1 if K1 > 0 else 0.0   # inherit K2/K1 ratio from params cell

K1_SWEEP = np.logspace(np.log10(100), np.log10(3000000), 60)  # N/m

# Null depth threshold: minima shallower than this fraction of peak are flagged
# as unreliable (the resonance null is ill-defined for very shallow minima)
NULL_DEPTH_THRESH = 0.03   # 3 % of profile max

sweep_f_cr           = []
sweep_x_null_res     = []
sweep_x_null_ES      = []
sweep_null_res_valid = []
sweep_null_ES_valid  = []

opts_sw  = SolverOptions(n_points=600, g=G_AIR, method='analytical')
obs_ref  = cant.L - 100e-6   # reference obs point for f_CR identification

# Initial search window -- broad on the first pass
f_lo, f_hi = 80e3, 560e3
f_prev = None

print(f'K1 sweep: {len(K1_SWEEP)} values  {K1_SWEEP[0]:.0f} -> {K1_SWEEP[-1]:.0f} N/m  K2_FACTOR={K2_FACTOR:.3f}...')

for i_k, k1_val in enumerate(K1_SWEEP):
    ct = ContactParams(k1=k1_val, k2=K2_FACTOR*k1_val, c1=C1, c2=0.0)

    # --- find f_CR: coarse 1 kHz amplitude peak (robust to matrix
    # ill-conditioning), then bounded minimize_scalar refinement (linewidth ~Hz) ---
    opts_id = SolverOptions(n_points=300, g=G_AIR, method='analytical',
                            f_start=f_lo, f_stop=f_hi, f_step=1e3)
    sw_id   = frequency_sweep(cant, ct, exc, opts_id, obs_x=obs_ref)
    i_pk    = int(np.argmax(sw_id['amplitudes_disp']))
    f_pk    = sw_id['freqs'][i_pk]
    _ref_k  = minimize_scalar(
        lambda f: -solve_single_frequency(f, cant, ct, exc, opts_pt,
                                          obs_x=obs_ref)['A_disp'],
        bounds=(f_pk - 1.2e3, f_pk + 1.2e3),
        method='bounded', options=dict(xatol=0.05))
    f_cr_k  = float(_ref_k.x)
    sweep_f_cr.append(f_cr_k)

    # Narrow the window for next iteration: +-20 kHz around current f_CR
    # (contact resonance varies smoothly with k1)
    f_lo = max(80e3,  f_cr_k - 20e3)
    f_hi = min(600e3, f_cr_k + 20e3)

    # --- resonance null from profile at f_CR ---
    res_k   = solve_single_frequency(f_cr_k, cant, ct, exc, opts_sw)
    xg_k, zg_k = res_k['x'], res_k['z']
    A_res_k = np.array([abs(complex(
        float(np.interp(xo, xg_k, np.real(zg_k))),
        float(np.interp(xo, xg_k, np.imag(zg_k))))) for xo in obs_pos])
    A_res_norm = A_res_k / A_res_k.max() if A_res_k.max() > 0 else A_res_k

    i_r = int(np.argmin(A_res_k))
    lo_r, hi_r = max(0, i_r - 5), min(N_OBS - 1, i_r + 5)
    null_is_deep = A_res_norm[i_r] < NULL_DEPTH_THRESH
    try:
        a_, b_, _ = np.polyfit(dist_tip[lo_r:hi_r+1], A_res_k[lo_r:hi_r+1], 2)
        xn_r    = -b_ / (2*a_) if a_ > 0 else dist_tip[i_r]
        valid_r = (a_ > 0
                   and dist_tip.min() < xn_r < dist_tip.max()
                   and 0 < i_r < N_OBS - 1
                   and null_is_deep)
    except Exception:
        xn_r, valid_r = dist_tip[i_r], False
    sweep_x_null_res.append(float(np.clip(xn_r, dist_tip.min(), dist_tip.max())))
    sweep_null_res_valid.append(valid_r)

    # --- ES blind spot from quasi-static profile ---
    exc_es_k = ExcitationParams(d1=0.0, d2=0.0, f0=0.0,
                                use_f0=False, bias_coupled_f0=False,
                                delta_V=DELTA_V, V_AC=V_AC, eps_r=1.0,
                                use_distributed_load=True)
    res_es_k = solve_single_frequency(F_QUASI, cant, ct, exc_es_k, opts_sw)
    xg_e, zg_e = res_es_k['x'], res_es_k['z']
    A_es_k = np.array([abs(complex(
        float(np.interp(xo, xg_e, np.real(zg_e))),
        float(np.interp(xo, xg_e, np.imag(zg_e))))) for xo in obs_pos])
    A_es_norm = A_es_k / A_es_k.max() if A_es_k.max() > 0 else A_es_k

    i_e = int(np.argmin(A_es_k))
    lo_e, hi_e = max(0, i_e - 5), min(N_OBS - 1, i_e + 5)
    es_is_deep = A_es_norm[i_e] < NULL_DEPTH_THRESH
    try:
        a_, b_, _ = np.polyfit(dist_tip[lo_e:hi_e+1], A_es_k[lo_e:hi_e+1], 2)
        xn_e    = -b_ / (2*a_) if a_ > 0 else dist_tip[i_e]
        valid_e = (a_ > 0
                   and dist_tip.min() < xn_e < dist_tip.max()
                   and 0 < i_e < N_OBS - 1
                   and es_is_deep)
    except Exception:
        xn_e, valid_e = dist_tip[i_e], False
    sweep_x_null_ES.append(float(np.clip(xn_e, dist_tip.min(), dist_tip.max())))
    sweep_null_ES_valid.append(valid_e)

    if i_k % 5 == 0:
        print(f'  k1={k1_val:6.0f} N/m  f_CR={f_cr_k/1e3:.1f} kHz  '
              f'x_res={sweep_x_null_res[-1]:.2f} um ({"OK" if valid_r else "edge/shallow"})  '
              f'x_ES={sweep_x_null_ES[-1]:.2f} um ({"OK" if valid_e else "edge/shallow"})')

sweep_f_cr           = np.array(sweep_f_cr)
sweep_x_null_res     = np.array(sweep_x_null_res)
sweep_x_null_ES      = np.array(sweep_x_null_ES)
sweep_null_res_valid = np.array(sweep_null_res_valid)
sweep_null_ES_valid  = np.array(sweep_null_ES_valid)
sweep_Dx             = sweep_x_null_ES - sweep_x_null_res
tip_offset_um        = cant.tip_offset * 1e6
print('Done.')

In [ ]:
# Valid-only masks (interior minimum, not clamped to scan edge)
k1_vr = K1_SWEEP[sweep_null_res_valid]
k1_ve = K1_SWEEP[sweep_null_ES_valid]
k1_vb = K1_SWEEP[sweep_null_res_valid & sweep_null_ES_valid]

fig_sw = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'f_CR vs k1  (branch-hopping diagnostic)',
        'Null positions vs k1',
        '',
        'Offset Dx = x_ES - x_res vs k1  (valid points only)',
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.14,
)

# ── Top-left: f_CR diagnostic ─────────────────────────────────────────────────
fig_sw.add_trace(go.Scatter(
    x=K1_SWEEP, y=sweep_f_cr * 1e-3,
    mode='lines+markers', marker=dict(size=5),
    line=dict(color='#70ad47', width=2),
    name='f_CR (det minimiser)',
), row=1, col=1)
fig_sw.add_vline(x=K1, line_dash='dot', line_color='green', line_width=1,
                 annotation_text=f'current k1', row=1, col=1)
fig_sw.update_yaxes(title_text='f_CR (kHz)', row=1, col=1)

# ── Top-right: null positions ─────────────────────────────────────────────────
# All points (faded)
fig_sw.add_trace(go.Scatter(
    x=K1_SWEEP, y=sweep_x_null_res,
    mode='markers', marker=dict(size=4, color='#1f4e79', opacity=0.25),
    showlegend=False,
), row=1, col=2)
fig_sw.add_trace(go.Scatter(
    x=K1_SWEEP, y=sweep_x_null_ES,
    mode='markers', marker=dict(size=4, color='#c00000', opacity=0.25),
    showlegend=False,
), row=1, col=2)
# Valid points only (solid)
fig_sw.add_trace(go.Scatter(
    x=k1_vr, y=sweep_x_null_res[sweep_null_res_valid],
    mode='lines+markers', marker=dict(size=5),
    line=dict(color='#1f4e79', width=2.5),
    name='Resonance null (valid)',
), row=1, col=2)
fig_sw.add_trace(go.Scatter(
    x=k1_ve, y=sweep_x_null_ES[sweep_null_ES_valid],
    mode='lines+markers', marker=dict(size=5),
    line=dict(color='#c00000', width=2.5, dash='dot'),
    name=f'ES blind spot (valid)',
), row=1, col=2)
if tip_offset_um > 0:
    fig_sw.add_hline(y=tip_offset_um, line_dash='dash', line_color='gray',
                     line_width=1,
                     annotation_text=f'contact pt ({tip_offset_um:.0f} um)',
                     annotation_font_color='gray', row=1, col=2)
fig_sw.add_vline(x=K1, line_dash='dot', line_color='green', line_width=1,
                 row=1, col=2)
fig_sw.update_yaxes(title_text='Distance from physical tip (um)', row=1, col=2)

# ── Bottom-right: Dx (valid only) ─────────────────────────────────────────────
Dx_valid = sweep_Dx[sweep_null_res_valid & sweep_null_ES_valid]
fig_sw.add_trace(go.Scatter(
    x=k1_vb, y=Dx_valid,
    mode='lines+markers',
    marker=dict(size=6,
                color=['#c00000' if d < 0 else '#1f4e79' for d in Dx_valid]),
    line=dict(color='#7030a0', width=2.5),
    name='Dx (valid)',
), row=2, col=2)
fig_sw.add_hline(y=0, line_dash='dash', line_color='gray',
                 line_width=0.8, row=2, col=2)
fig_sw.add_vline(x=K1, line_dash='dot', line_color='green', line_width=1,
                 row=2, col=2)
fig_sw.update_yaxes(title_text='Dx (um)', row=2, col=2)

for row in (1, 2):
    for col in (1, 2):
        fig_sw.update_xaxes(title_text='k1 (N/m)', type='log', row=row, col=col)

fig_sw.update_layout(
    height=720, width=1140,
    template='plotly_white',
    title='Contact stiffness sweep — null positions and Dx (faded = minimum at scan edge)',
    legend=dict(x=0.02, y=0.50, font_size=11),
)
fig_sw.show()

fname = os.path.join(RESULTS_DIR, 'nb07_k1_sweep')
fig_sw.write_html(fname  + '.html')
print(f'Saved: {os.path.basename(fname)}.html')


---
## Part 5 — Spatial profiles vs contact stiffness (load)

For each k₁ in the sweep, plot the full displacement and slope profiles |z(x)| and |z’(x)|.
Left panels: quasi-static (30 kHz, body ES only). Right panels: on-resonance (f_CR).
Colour encodes amplitude; vertical dashed line marks the contact point.

In [ ]:
# Pick a subset of k1 values evenly spaced in log scale
N_SUBSET   = 6
idx_subset = np.round(np.linspace(0, len(K1_SWEEP) - 1, N_SUBSET)).astype(int)
k1_subset  = K1_SWEEP[idx_subset]
fcr_subset = sweep_f_cr[idx_subset]

opts_pr = SolverOptions(n_points=500, g=G_AIR, method='analytical')

# Colour ramp from light to dark for increasing load
import plotly.colors as pc
ramp = pc.sample_colorscale('Blues', [0.35 + 0.65*i/(N_SUBSET-1)
                                       for i in range(N_SUBSET)])

fig_pr = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'Displacement |z(x)|  quasi-static ({F_QUASI/1e3:.0f} kHz, ES only)',
        'Displacement |z(x)|  on-resonance (f_CR)',
        f"Slope |z'(x)|  quasi-static ({F_QUASI/1e3:.0f} kHz, ES only)",
        "Slope |z'(x)|  on-resonance (f_CR)",
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.12,
)

exc_es_pr = ExcitationParams(
    d1=0.0, d2=0.0, f0=0.0,
    use_f0=False, bias_coupled_f0=False,
    delta_V=DELTA_V, V_AC=V_AC, eps_r=1.0,
    use_distributed_load=True,
)

for i, (k1_val, f_cr_k, color) in enumerate(zip(k1_subset, fcr_subset, ramp)):
    # k2 must match the sweep that produced fcr_subset (K2_FACTOR*k1);
    # with k2=0 the profiles would be far off-resonance at high k1
    ct  = ContactParams(k1=k1_val, k2=K2_FACTOR*k1_val, c1=C1, c2=0.0)
    lbl = f'k1={k1_val:.0f} N/m'
    show_leg = True

    # quasi-static ES only
    res_e = solve_single_frequency(F_QUASI, cant, ct, exc_es_pr, opts_pr)
    x_um  = res_e['x'] * 1e6
    z_e   = np.abs(res_e['z'])
    zp_e  = np.abs(res_e['zp'])

    # on-resonance (combined excitation as set by flags)
    res_r = solve_single_frequency(f_cr_k, cant, ct, exc, opts_pr)
    z_r   = np.abs(res_r['z'])
    zp_r  = np.abs(res_r['zp'])

    for row, (y_e, y_r, units) in enumerate([
        (z_e  * 1e12, z_r  * 1e12, 'pm'),
        (zp_e,        zp_r,        'm/m'),
    ], start=1):
        for col, y_vals in enumerate([y_e, y_r], start=1):
            fig_pr.add_trace(go.Scatter(
                x=x_um, y=y_vals,
                mode='lines',
                line=dict(color=color, width=2),
                name=lbl,
                showlegend=(row == 1 and col == 1),
                legendgroup=lbl,
            ), row=row, col=col)

# Contact point marker on all panels
for row in (1, 2):
    for col in (1, 2):
        fig_pr.add_vline(
            x=cant.contact_x * 1e6,
            line_dash='dash', line_color='gray', line_width=1.2,
            annotation_text=f'contact ({cant.contact_x*1e6:.0f} um)',
            annotation_font_color='gray', annotation_font_size=9,
            row=row, col=col,
        )
        fig_pr.update_xaxes(
            title_text='Position along cantilever (um)', row=row, col=col)

fig_pr.update_yaxes(title_text='|z(x)| (pm)',   row=1, col=1)
fig_pr.update_yaxes(title_text='|z(x)| (pm)',   row=1, col=2)
fig_pr.update_yaxes(title_text="|z'(x)| (m/m)", row=2, col=1)
fig_pr.update_yaxes(title_text="|z'(x)| (m/m)", row=2, col=2)

fig_pr.update_layout(
    height=660, width=1140,
    template='plotly_white',
    title=(
        f'Spatial profiles for {N_SUBSET} contact stiffness values  |  '
        f'tip_offset={cant.tip_offset*1e6:.0f} um  '
        f'(darker = higher k1 / load)'
    ),
    legend=dict(x=1.02, y=0.98, font_size=11,
                bordercolor='lightgray', borderwidth=1),
)
fig_pr.show()

fname = os.path.join(RESULTS_DIR, 'nb07_profiles_vs_k1')
fig_pr.write_html(fname  + '.html')
print(f'Saved: {os.path.basename(fname)}.html')


---
## Part 6 — Mode-shape contributions: ES, Piezo, and Combined

Solve at the **current** k₁ and f_CR under three excitation conditions:
- **ES only** (body electrostatics, no piezo)
- **Piezo only** (substrate drive d₁, no electrostatics)
- **Combined** (both active simultaneously)

Two frequencies: **off-resonance** (30 kHz) and **on-resonance** (f_CR).  
Two observables: **displacement** |z(x)| and **slope** |z’(x)|.  
The contact point is marked with a dashed vertical line.

In [ ]:
opts_ms = SolverOptions(n_points=600, g=G_AIR, method='analytical')

exc_es_ms = ExcitationParams(
    d1=0.0, d2=0.0, f0=0.0,
    use_f0=False, bias_coupled_f0=False,
    delta_V=DELTA_V, V_AC=V_AC, eps_r=1.0,
    use_distributed_load=True,
)
exc_piezo_ms = ExcitationParams(
    d1=D1, d2=0.0, f0=0.0,
    use_f0=False, bias_coupled_f0=False,
    delta_V=0.0, V_AC=0.0, eps_r=1.0,
    use_distributed_load=False,
)
exc_both_ms = ExcitationParams(
    d1=D1, d2=0.0, f0=0.0,
    use_f0=False, bias_coupled_f0=False,
    delta_V=DELTA_V, V_AC=V_AC, eps_r=1.0,
    use_distributed_load=True,
)

cases_ms = [
    ('ES only',     exc_es_ms,    '#c00000', 'solid'),
    ('Piezo only',  exc_piezo_ms, '#1f4e79', 'solid'),
    ('Piezo + ES',  exc_both_ms,  '#2e75b6', 'dash'),
]

freqs_ms = [('Off-resonance', F_QUASI), ('On-resonance', F_CR)]

# Solve all combinations
results_ms = {}
for freq_label, freq_val in freqs_ms:
    results_ms[freq_label] = {}
    for case_label, exc_ms, _, _ in cases_ms:
        results_ms[freq_label][case_label] = solve_single_frequency(
            freq_val, cant, contact, exc_ms, opts_ms)

# ── Plot ──────────────────────────────────────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_ms = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'Slope |z\u2019(x)|  Off-resonance ({F_QUASI/1e3:.0f} kHz)',
        f'Slope |z\u2019(x)|  On-resonance ({F_CR/1e3:.0f} kHz)',
        'Displacement |z(x)|  Off-resonance',
        'Displacement |z(x)|  On-resonance',
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.12,
)

for col, (freq_label, freq_val) in enumerate(freqs_ms, start=1):
    for case_label, exc_ms, color, dash in cases_ms:
        res = results_ms[freq_label][case_label]
        x_um = res['x'] * 1e6
        zp_abs = np.abs(res['zp'])
        z_abs  = np.abs(res['z'])

        show_leg = (col == 1)
        fig_ms.add_trace(go.Scatter(
            x=x_um, y=zp_abs,
            mode='lines',
            line=dict(color=color, width=2.2, dash=dash),
            name=case_label, showlegend=show_leg,
            legendgroup=case_label,
        ), row=1, col=col)

        fig_ms.add_trace(go.Scatter(
            x=x_um, y=z_abs,
            mode='lines',
            line=dict(color=color, width=2.2, dash=dash),
            name=case_label, showlegend=False,
            legendgroup=case_label,
        ), row=2, col=col)

    # Contact point marker
    for row in (1, 2):
        fig_ms.add_vline(
            x=cant.contact_x * 1e6,
            line_dash='dash', line_color='gray', line_width=1.2,
            annotation_text=f'contact ({cant.contact_x*1e6:.0f} um)',
            annotation_font_color='gray', annotation_font_size=9,
            row=row, col=col,
        )

for col in (1, 2):
    fig_ms.update_yaxes(title_text="|z'(x)| (m/m)", row=1, col=col)
    fig_ms.update_yaxes(title_text='|z(x)| (m)',    row=2, col=col)
    for row in (1, 2):
        fig_ms.update_xaxes(title_text='Position along cantilever (um)',
                            row=row, col=col)

fig_ms.update_layout(
    height=640, width=1140,
    template='plotly_white',
    title=(
        f'Mode-shape contributions: ES / Piezo / Combined  |  '
        f'k1={K1:.0f} N/m  |  f_CR={F_CR/1e3:.1f} kHz'
    ),
    legend=dict(x=0.01, y=0.99, font_size=12,
                bordercolor='lightgray', borderwidth=1),
)
fig_ms.show()

fname = os.path.join(RESULTS_DIR, 'nb07_mode_shape_contributions')
fig_ms.write_html(fname  + '.html')
print(f'Saved: {os.path.basename(fname)}.html')


---
## Discussion

### What the corrected two-segment model changes

Earlier versions of this notebook used the legacy terminal-BC + cubic-overhang
treatment, which manufactured spurious structure on precisely the stretch of beam
where the near-tip nulls live (see `docs/model.md` and notebook 09 Section 2). With the corrected
solver:

- **The resonance null (D-NS) is a genuine interior node on the overhang.** At the
  default k₁ = 3 kN/m (Mindlin k₂ = 0.85 k₁) it sits ≈7.7 µm from the physical
  tip, i.e. ≈2.3 µm *past* the contact point. With increasing k₁ it approaches the
  contact from the overhang side and saturates ≈0.6–0.9 µm past it — it never
  reaches the contact point. This is the rigid-Mindlin saturation ξ* ∝ k₂H²·F/M
  (cf. ≈3.6 µm for the H = 15 µm geometry of notebook 09; here H = 6 µm).
- **There is no interior electrostatic blind spot at soft-to-moderate contact.**
  The distributed ES load also acts on the overhang — the part of the lever
  closest to the sample, taking ≈2.3× the load per unit length of the region above
  the contact. At k₁ = 3 kN/m the 30 kHz ES response decreases monotonically
  toward the tip (tip/edge ratio ≈ 0.035) but has **no zero**; the extracted
  "blind spot" is the profile minimum at the scan edge (the notebook now flags
  this). An interior ESBS first appears near k₁ ≈ 4 kN/m for this geometry and
  k₂ model (within 0.5 µm of the tip, migrating inward with k₁). Under the legacy solver this notebook reported an interior blind spot
  here — that was an artifact of omitting the overhang load.
- **Δx at this k₁ is not a null-to-null distance.** Δx = −7.7 µm measures the
  distance from the resonance node to the free end where the ES minimum sits.

### Null positions vs load (Part 4)

The D-NS enters the scan window at k₁ ≈ 0.6 kN/m, crosses onto the overhang, and
saturates ≈0.6–0.9 µm past the contact (9.1–9.4 µm from the tip). The interior
ESBS exists only above k₁ ≈ 4 kN/m; it approaches the contact point slowly from
the tip side, so at experimentally realistic stiffnesses (≲ 30 kN/m) the two nulls
remain separated by ≈3–6 µm — the persistent D-NS/ESBS separation of the
rigid-Mindlin lateral-coupling model. (With k₂ = 0 both nulls would instead
converge to contact − H·tanφ ≈ 1.2 µm behind the contact; the k₂ model is the
qualitative switch, see notebook 09.) The dip in the ES curve near
k₁ ~ 0.5 MN/m is the global minimum hopping between two local minima of
comparable depth, not a discontinuous physical migration — far beyond
experimentally accessible contact stiffness in any case.

### The spot where the resonance doesn't matter

The 2D spectrogram shows the resonance amplitude reaching a deep minimum at the
D-NS near the tip. Placing an interferometer laser here minimises the driven
resonance signal regardless of excitation source (piezo, body ES, or both) — this
is the **displacement null** of mode 1. The anti-resonance structure visible in
the log-scale heatmap arises from destructive interference between the piezo
drive (coupling through the tip spring) and the body-ES drive (coupling through
the distributed load).

### Practical implication for PFM

| Laser position | Effect at *f*_CR | Effect at 30 kHz |
|----------------|-----------------|------------------|
| *x*_null (resonance null) | min resonance signal | body-ES signal **not** zeroed |
| *x*_ES_blind (ES blind spot, k₁ ≳ 4 kN/m only) | resonance signal retained | min body-ES signal |

Two protocol consequences: (i) a blind-spot "parking" calibration is only
well-defined at hard contact — below k₁ ≈ 4 kN/m there is nothing to park on;
(ii) because the D-NS saturates *past* the contact while the ESBS approaches it,
the two conditions cannot be satisfied simultaneously at any load for
rigid-Mindlin lateral coupling.

### Parameter sweeps to try

| Change | Effect |
|--------|--------|
| Increase `K1` | D-NS saturates ≈0.6–0.9 µm past contact; interior ESBS appears above ≈4 kN/m |
| Set `K2 = 0` | Both nulls converge to contact − H·tanφ; ESBS onset moves to lower k₁ |
| Toggle `USE_PIEZO=False` | Spectrogram changes; null position unchanged |
| Change `F_QUASI` | ES blind spot shifts; resonance null fixed |
| Increase `cant.tip_offset` | Longer overhang: larger overhang load share, stronger blind-spot suppression |